In [ ]:
# import sys
# print(sys.executable)

In [ ]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings

## Indexing

In [ ]:
# WebBAseLoader -> downloads given URL and parses the HTML
# load documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()
docs

In [ ]:
# Split the document into chunks
# chunk_size=1000: target max size of each chunk (in characters).
# chunk_overlap=200: each chunk overlaps the previous by 200 characters to preserve context at boundaries.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
splits

In [ ]:
# printing each chunk size
for idx, i in enumerate(splits):
    print(idx, len(i.page_content))

In [ ]:
# Embed
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") not using because of ratelimit
# using huggingface
vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()